## [Evaluamos al profesor Marco Cañas Aquí](https://forms.office.com/Pages/ResponsePage.aspx?id=IefhmYRxjkmK_7KtTlPBwkanXIs1i1FEujpsZgO6dXpUREJPV1kxUk1JV1ozTFJIQVNIQjY5WEY3US4u)

# [Vínculo al formulario de google del diagnóstico](https://forms.gle/LXCJMWhBQjfHGEXu8)

Aquí tienes el script de Python que genera el archivo Excel solicitado, realizando la comparación difusa de nombres entre ambos archivos y conservando la estructura del primer archivo con los estudiantes que realizaron el examen.


In [17]:
import pandas as pd
import unicodedata
from rapidfuzz import fuzz

# --- Configuración de rutas ---
ruta_primer_archivo = "2_bijao_grupo 4_final.xlsx"
ruta_segundo_archivo = "4_bijao_grupo_4_diagnostica_y_final.xls"
ruta_salida = "3_bijao_grupo_4_final_ordenado.xlsx"

# --- 1. Cargar los datos ---
df_original = pd.read_excel(ruta_primer_archivo, sheet_name=0)
df_referencia = pd.read_excel(ruta_segundo_archivo, sheet_name="P.I")

# --- 2. Limpiar nombres (minúsculas, sin tildes, sin espacios extras) ---
def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return ""
    nombre = str(nombre).lower().strip()
    nombre = unicodedata.normalize("NFKD", nombre).encode("ASCII", "ignore").decode("ASCII")
    return nombre

# Construir nombres completos en df_original (primer archivo)
df_original["Nombre completo"] = (
    df_original["Student First Name"].fillna("").astype(str) + " " +
    df_original["Student Last Name"].fillna("").astype(str)
)
df_original["Nombre completo limpio"] = df_original["Nombre completo"].apply(limpiar_nombre)

# Extraer nombres de referencia desde hoja "P.I" (segundo archivo)
# La columna de nombres está en la fila 4 (índice 4) y columna 2 (índice 2)
nombres_referencia = []
for idx, row in df_referencia.iterrows():
    if idx >= 4:  # Empezar desde el índice 5 (fila 5 en Excel = índice 4 en pandas)
        nombre = row.iloc[2]  # columna C (índice 2)
        if pd.notna(nombre):
            nombres_referencia.append(str(nombre).strip())

df_ref_nombres = pd.DataFrame({"Nombre y Apellido": nombres_referencia})
df_ref_nombres["Nombre y Apellido limpio"] = df_ref_nombres["Nombre y Apellido"].apply(limpiar_nombre)

# --- 3. Emparejar usando similitud de cadenas ---
# Crear diccionario para mapear nombre limpio de referencia a su nombre original
mapa_referencia = dict(zip(df_ref_nombres["Nombre y Apellido limpio"], df_ref_nombres["Nombre y Apellido"]))

# Función para encontrar mejor coincidencia
def encontrar_mejor_coincidencia(nombre_limpio, df_referencia_limpios, umbral=80):
    mejor_puntaje = 0
    mejor_nombre = None
    for nombre_ref_limpio in df_referencia_limpios:
        puntaje = fuzz.ratio(nombre_limpio, nombre_ref_limpio)
        if puntaje > mejor_puntaje and puntaje >= umbral:
            mejor_puntaje = puntaje
            mejor_nombre = nombre_ref_limpio
    return mejor_nombre

# Obtener lista de nombres limpios de referencia
nombres_ref_limpios = df_ref_nombres["Nombre y Apellido limpio"].tolist()

# Mapear cada nombre original con su mejor coincidencia en referencia
mapeo_coincidencias = {}
for nombre_limpio in df_original["Nombre completo limpio"].unique():
    coincidencia = encontrar_mejor_coincidencia(nombre_limpio, nombres_ref_limpios)
    if coincidencia:
        mapeo_coincidencias[nombre_limpio] = coincidencia

# Crear columna de coincidencia en df_original
df_original["Nombre Referencia"] = df_original["Nombre completo limpio"].map(mapeo_coincidencias)

# Filtrar solo estudiantes que coincidieron
df_final = df_original[df_original["Nombre Referencia"].notna()].copy()

# --- 4. Crear columna "Nombre y Apellido" con el nombre de referencia ---
df_final["Nombre y Apellido"] = df_final["Nombre Referencia"]

# Reorganizar columnas: poner "Nombre y Apellido" al inicio
columnas = ["Nombre y Apellido"] + [col for col in df_final.columns if col not in ["Nombre y Apellido", "Nombre completo", "Nombre completo limpio", "Nombre Referencia"]]
df_final = df_final[columnas]

# --- 5. Guardar archivo final ---
df_final.to_excel(ruta_salida, index=False)
print(f"Archivo generado: {ruta_salida}")
print(f"Total estudiantes en archivo final: {len(df_final)}")


Archivo generado: 3_bijao_grupo_4_final_ordenado.xlsx
Total estudiantes en archivo final: 46


# Explicación del script

### 1. **Carga de archivos**
- Lee el primer archivo `2_bijao_grupo_4_final.xlsx` (con las respuestas).
- Lee la hoja `P.I` del segundo archivo Excel (de referencia).

### 2. **Limpieza de nombres**
- Se crea el nombre completo en el primer archivo: `Student First Name` + `Student Last Name`.
- Se extraen los nombres de la columna `C` de la hoja `P.I` a partir de la **fila 5** (índice 4).
- Se normalizan todos los nombres:
  - Minúsculas
  - Eliminación de tildes (usando `unicodedata`)
  - Eliminación de espacios extra

### 3. **Coincidencia difusa**
- Usa la librería `rapidfuzz` para calcular similitud entre cadenas.
- Cada nombre del primer archivo se compara con todos los nombres de referencia.
- Si la similitud es ≥ **80%**, se considera una coincidencia.

### 4. **Filtrado y creación del archivo final**
- Solo se conservan las filas del primer archivo que tuvieron coincidencia.
- Se crea una nueva columna `Nombre y Apellido` con el nombre exacto tomado del archivo de referencia.
- Se eliminan las columnas auxiliares usadas en el proceso.

### 5. **Guardado**
- El archivo se guarda como `2_bijao_grupo_4_final_corregido.xlsx`.

## 📦 Instalación de dependencias

```bash
pip install pandas openpyxl xlrd rapidfuzz
```

## ⚠️ Notas importantes

- El umbral del 80% es configurable; puedes ajustarlo según la calidad de los datos.
- Si un nombre no alcanza el umbral, se excluye del archivo final (su fila queda vacía en el primer archivo, pero no se incluye en la salida porque el requerimiento pide solo estudiantes que realizaron el examen).